# DL_04: Hybrid Probability Fusion, Validation Selection & Graph Integration
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook integrates Phase 1 Applied Machine Learning predictions with Phase 2 Deep Learning predictions into a unified neuro-symbolic causal reasoning pipeline.

Key requirements:
- Ingest aligned AML and PubMedBERT predictions for **both** validation (1,914 rows) and test (1,408 rows) splits.
- Explicitly validate candidate alignment with strict assertions against duplicate IDs, missing candidates, and mismatched labels.
- Implement validation-selected linear probability-score fusion: $P_{\text{hybrid}} = \alpha \cdot P_{\text{DL}} + (1 - \alpha) \cdot P_{\text{AML}}$.
- Calibrate the fusion parameter $\alpha \in [0, 1]$ and decision threshold $\tau$ strictly on the held-out validation predictions (4 PMC articles). No test-set data may influence parameter selection.
- Primary AML baseline: `Logistic Regression (Moderate Weight 1:2)` fitted on the training split, with alternative models reported as baselines.
- Apply linguistic sanity validation (negation filtering, distance constraints).
- Preserve the standardized 17-column `UnifiedRelationRecord` schema.
- Reuse Phase 1 graph functionality from `AML/causal_extractors.py` (`enforce_dag_acyclicity`).
- Compile acyclic DAGs into `pgmpy` Bayesian networks for interventional queries.
- **CRITICAL CONSTRAINT**: Corr2Cause is an abstract NLI benchmark and must NOT feed into the BioCause knowledge graph.


## 2. Experimental Methodology & Theoretical Background

### Hybrid Ensembling & Validation Discipline
- **AML Stream**: Syntactic dependency path (SDP) and lexical trigger features provide high precision on canonical syntactic patterns.
- **DL Stream (PubMedBERT Fulltext)**: Captures complex biomedical context and implicit causal expressions.
- **Validation Discipline**: $\alpha^*$ is selected by sweeping $\alpha \in [0.0, 1.0]$ with step 0.05 strictly on the 4 held-out validation articles (1,914 candidates). Once $\alpha^*$ is frozen, it is applied to the test split.
- **Evaluation**: Probability-score fusion is empirically compared against both pure AML ($\alpha = 0$) and pure DL ($\alpha = 1$) baselines.

### Symbolic Graph Compilation & Interventional Reasoning
- Extracted relations are converted into directed edges $(u \to v)$ in NetworkX.
- The deterministic cycle breaker (`enforce_dag_acyclicity`) prunes self-loops, resolves bidirectional conflicts, and breaks feedback cycles using confidence scores.
- **Causal Validity Guardrail**: An acyclic graph is a topological property, not an absolute guarantee of causal ground truth. Cycle breaking enforces formal mathematical consistency required for Bayesian inference.
- Compiled Bayesian networks in `pgmpy` compute observational distributions $P(Y \mid X)$ and Pearl's interventional distributions $P(Y \mid do(X))$. Demonstration interventional queries illustrate formal calculus capabilities.


## 3. Expected Inputs & Prediction Contracts

| Stream | File Path | Expected Rows | Required Columns |
| :--- | :--- | :---: | :--- |
| **AML Validation Preds** | `data/processed/aml_biocause_validation_predictions.csv` | 1,914 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **AML Test Preds** | `data/processed/aml_biocause_test_predictions.csv` | 1,408 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **DL Validation Preds** | `data/processed/dl_biocause_validation_predictions.csv` | 1,914 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **DL Test Preds** | `data/processed/dl_biocause_test_predictions.csv` | 1,408 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **BioCause Candidates** | `data/processed/biocause_candidates.csv` | 13,408 | 17 metadata columns |

> **AML Prerequisite Note**: If `aml_biocause_*.csv` files are not present in `data/processed/`, they must be produced by executing the unchanged Phase 1 classifier pipeline from `AML/AML_04_classical_classifier.ipynb` fitted on the training split to output candidate-aligned held-out validation and test predictions.


## 4. Technical Responsibilities & Implementation Requirements

1. **Portable Repo Setup**: Discover root and import `dl_utils` and `causal_extractors`.
2. **Dual Split Alignment & Validation**: Ingest validation and test files; assert candidate ID alignment, row counts, and matching ground-truth labels.
3. **Validation-Only Optimization**: Grid-search $\alpha \in [0, 1]$ and threshold $\tau$ strictly on validation articles to maximize validation F1.
4. **Linguistic Sanity Filter**: Apply negation pruning and distance filtering.
5. **17-Column Schema Formatting**: Map positive relations into `dl_utils.UNIFIED_RELATION_SCHEMA`.
6. **NetworkX DAG Compilation**: Build directed graph and call `enforce_dag_acyclicity` from `AML/causal_extractors.py`.
7. **Edge Evaluation**: Compute graph edge precision, recall, and F1 before and after cycle breaking.
8. **Bayesian Network Compilation**: Construct `pgmpy` `DiscreteBayesianNetwork` and execute illustrative $do$-calculus queries.
9. **Artifact Export**: Export hybrid predictions and graph structures for `DL_05`.


In [ ]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
from pathlib import Path

def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
from causal_extractors import enforce_dag_acyclicity
import networkx as nx
import pandas as pd
import numpy as np
print(f'Repository root located at: {ROOT_DIR}')


In [ ]:
# Section 2: Ingestion & Alignment of AML and DL Streams (Validation & Test)
# TODO: Load AML and DL validation and test prediction CSVs
# AML Primary: dl_utils.PRIMARY_AML_CLASSIFIER_NAME ('Logistic Regression (Moderate Weight 1:2)')
# File paths:
# - data/processed/aml_biocause_validation_predictions.csv (1,914 rows)
# - data/processed/aml_biocause_test_predictions.csv (1,408 rows)
# - data/processed/dl_biocause_validation_predictions.csv (1,914 rows)
# - data/processed/dl_biocause_test_predictions.csv (1,408 rows)

# Strict Alignment Assertions:
# assert len(df_aml_val) == len(df_dl_val) == 1914
# assert len(df_aml_test) == len(df_dl_test) == 1408
# assert (df_aml_val['candidate_id'].values == df_dl_val['candidate_id'].values).all()
# assert (df_aml_val['true_label'].values == df_dl_val['true_label'].values).all()
# assert (df_aml_test['candidate_id'].values == df_dl_test['candidate_id'].values).all()
# assert (df_aml_test['true_label'].values == df_dl_test['true_label'].values).all()


In [ ]:
# Section 3: Validation-Selected Probability-Score Fusion
# TODO: Grid search alpha in [0.0, 1.0] (step 0.05) on validation split only
# Formula: P_hybrid = alpha * P_DL + (1 - alpha) * P_AML
# Select alpha* maximizing validation F1 score
# Select decision threshold tau* maximizing validation F1 score


In [ ]:
# Section 4: Linguistic Sanity Validation & Candidate Pruning
# TODO: Apply post-processing negation filters and distance checks on candidate between_text


In [ ]:
# Section 5: Standardized 17-Column UnifiedRelationRecord Formatting
# TODO: Format surviving positive predictions into dl_utils.UNIFIED_RELATION_SCHEMA


In [ ]:
# Section 6: NetworkX DAG Construction & Deterministic Cycle Breaking
# TODO: Construct directed graph and execute enforce_dag_acyclicity()
# - Log removed self-loops, bidirectional conflicts, and cycle-breaking edges
# - Compute edge precision, recall, and F1 before and after cycle pruning


In [ ]:
# Section 7: Bayesian Network Compilation & Interventional Queries
# TODO: Compile acyclic DAG into pgmpy DiscreteBayesianNetwork
# - Execute interventional query P(Y | do(X=1)) vs observational P(Y | X=1)


In [ ]:
# Section 8: Artifact Export
# TODO: Export data/processed/dl_biocause_hybrid_predictions.csv


## 5. Expected Outputs & Evaluation Criteria

- `data/processed/dl_biocause_hybrid_predictions.csv`: Standardized 17-column records.
- Optimal fusion parameters ($\alpha^*$, $\tau^*$) and validation search documentation.
- **Evaluation Criteria**: Comparative performance of hybrid model against pure AML and pure DL baselines, verified DAG acyclicity, and edge precision/recall/F1 before and after cycle breaking.
